# 02 — Fine-tune Prithvi-EO-2.0-300M (Land Cover) → ONNX → Hugging Face

**GISGPT** · เปลี่ยนสมองจาก baseline เป็น Geospatial Foundation Model จริง

รันบน **Colab GPU** (`Runtime → Change runtime type → T4 GPU`)

ขั้นตอน: login HF → ดาวน์โหลด Sen4Map → เทรน `scripts/finetune.py` → ได้ ONNX → push กลับ HF → เอาไปวางในแอป local

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '⚠️ ไม่มี GPU — เปิด Runtime → Change runtime type → T4')

## 1) ติดตั้ง dependencies

In [ ]:
!pip install -q terratorch onnx onnxscript h5py huggingface_hub requests

## 2) Login Hugging Face

โมเดล **Prithvi-EO-2.0-300M เปิดให้ดาวน์โหลดฟรี (Apache-2.0, ไม่ gated)** — ไม่ต้องกดยอมรับ license ใดๆ
แต่ต้อง login เพื่อ **push โมเดลที่เทรนแล้วกลับขึ้น HF**

สร้าง token: https://huggingface.co/settings/tokens (สิทธิ์ Write) แล้วรัน cell ด้านล่าง (token ไม่ต้องเอามาใส่ในโค้ด — ใส่ในช่อง prompt ที่ขึ้นมา)

In [ ]:
from huggingface_hub import login
login()  # ใส่ token ตรงนี้

## 3) ดาวน์โหลดข้อมูล Sen4Map (train/val)

ค้นหา link .h5 อัตโนมัติจากเว็บของ dataset (https://datapub.fz-juelich.de/sen4map/)
ถ้าไฟล์ใหญ่เกิน quota Colab ให้ใช้ `--subset` ลดจำนวน หรือดาวน์โหลด country-wise จากหน้าเว็บแทน

In [ ]:
import os, re, requests

def find_h5_links(page):
    r = requests.get(page, timeout=30)
    return sorted(set(re.findall(r'href=["\']([^"\']*\.h5)["\']', r.text)))

for sub in ['', 'train_val_test/']:
    links = find_h5_links('https://datapub.fz-juelich.de/sen4map/' + sub)
    if links:
        print(f'--- {sub or "root"} ---')
        for l in links:
            print(' ', l)
        break

In [ ]:
# ใส่ URL จริงของ train.h5 / val.h5 จากผลลัพธ์ข้างบน (หรือจากหน้าเว็บ)
TRAIN_URL = 'https://datapub.fz-juelich.de/sen4map/train_val_test/train.h5'  # ปรับตามจริง
VAL_URL   = 'https://datapub.fz-juelich.de/sen4map/train_val_test/val.h5'    # ปรับตามจริง

def download(url, out='data.h5'):
    if os.path.exists(out):
        print('มีอยู่แล้ว:', out); return out
    print('กำลังดาวน์โหลด', url)
    r = requests.get(url, stream=True, timeout=300)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    with open(out, 'wb') as fh:
        for chunk in r.iter_content(chunk_size=1 << 20):
            fh.write(chunk)
    print(f'✅ {out} ({os.path.getsize(out)/1e6:.0f} MB)')
    return out

train_h5 = download(TRAIN_URL, 'train.h5')
val_h5   = download(VAL_URL, 'val.h5')

## 4) อัปโหลด scripts/finetune.py ขึ้น Colab

ใช้เมนู Files (โฟลเดอร์ซ้ายมือ) ลากไฟล์ `GISGPT/scripts/finetune.py` มาวาง หรือใช้ cell ด้านล่าง

In [ ]:
from google.colab import files
uploaded = files.upload()  # เลือก finetune.py
assert 'finetune.py' in uploaded, 'ยังไม่ได้อัปโหลด finetune.py'

## 5) เทรน!

- `--subset 8000` = ใช้สุ่ม 8000 ตัวอย่าง (เร็ว ๆ ได้ สำหรับเล่น)
- `--push peeradon4778/prithvi-landcover-th` = push กลับ HF อัตโนมัติ (เปลี่ยนเป็น username ของตัวเอง)
- ถ้า terratorch/Prithvi โหลดไม่ผ่าน โค้ดจะ fallback เป็น CNN เล็กให้ pipeline ยังทำงานได้ — เอา error มาถามได้

In [ ]:
!python finetune.py --train train.h5 --val val.h5 --subset 8000 --epochs 8 --batch 16 --onnx prithvi_landcover.onnx --push peeradon4778/prithvi-landcover-th

## 6) ตรวจ ONNX (ต้องตรงกับที่แอปคาดหวัง)

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession('prithvi_landcover.onnx', providers=['CPUExecutionProvider'])
i, o = sess.get_inputs()[0], sess.get_outputs()[0]
print('input :', i.name, i.shape, i.type)
print('output:', o.name, o.shape, o.type)
assert list(i.shape) == [1, 1, 6, 224, 224], 'input ต้องเป็น (1,1,6,224,224)'
print('✅ ตรงกับสัญญาแอป GISGPT แล้ว')

## 7) เอาโมเดลไปใช้ในแอป local

1. ดาวน์โหลด `prithvi_landcover.onnx` + `class_names.json` (ที่ push ไว้) ลงเครื่อง
2. วางใน `GISGPT/models/` (ลบไฟล์ demo ทิ้ง: `prithvi_landcover_demo.*`)
3. รัน `python main.py` → แอปจะสลับเป็น Prithvi (ONNX) อัตโนมัติ

📥 ถ้าเทรนบน Colab แล้วปิด session ไปแล้ว โหลดกลับได้จาก:
`https://huggingface.co/peeradon4778/prithvi-landcover-th/resolve/main/prithvi_landcover.onnx`

In [ ]:
print('✅ จบ — ไปใช้ในแอปได้เลย!')